# Sistema Experto: Diagnóstico de Defectos en Soldadura
**Dominio:** `soldadura1`

### 1. Variables de Entrada (Predicados)
* `arco_inestable(x)`
* `porosidad_visible(x)`
* `salpicadura_excesiva(x)`
* `cordon_irregular(x)`
* `corriente_alta(x)`
* `avance_rapido(x)`
* `gas_insuficiente(x)`
* `material_sucio(x)`

### 2. Base de Conocimiento (Reglas Lógicas)
**Reglas de Nivel 1 (Hechos Intermedios)**
1. ∀x (arco_inestable(x) ∧ gas_insuficiente(x) → falla_cobertura_gas(x))
2. ∀x (salpicadura_excesiva(x) ∧ corriente_alta(x) → parametro_excedido(x))
3. ∀x (porosidad_visible(x) ∧ material_sucio(x) → contaminacion_base(x))
4. ∀x (cordon_irregular(x) ∧ avance_rapido(x) → tecnica_deficiente(x))

**Reglas de Nivel 2 (Encadenadas)**
5. ∀x (falla_cobertura_gas(x) ∧ porosidad_visible(x) → diagnostico_porosidad_gas(x))
6. ∀x (parametro_excedido(x) ∧ avance_rapido(x) → diagnostico_socavado_severo(x))
7. ∀x (contaminacion_base(x) ∧ falla_cobertura_gas(x) → diagnostico_rechazo_pieza(x))
8. ∀x (tecnica_deficiente(x) ∧ salpicadura_excesiva(x) → diagnostico_reentrenamiento(x))

In [1]:
!pip install SpeechRecognition
!pip install pyttsx3
!pip install PyAudio

In [ ]:
import speech_recognition as sr
import subprocess

# BASE DE CONOCIMIENTO (8 Variables, 8 Reglas encadenadas)


REGLAS = [
    # --- REGLAS DE NIVEL 1 ---
    {"condiciones": ["arco_inestable", "gas_insuficiente"], "conclusion": "falla_cobertura_gas"},
    {"condiciones": ["salpicadura_excesiva", "corriente_alta"], "conclusion": "parametro_excedido"},
    {"condiciones": ["porosidad_visible", "material_sucio"], "conclusion": "contaminacion_base"},
    {"condiciones": ["cordon_irregular", "avance_rapido"], "conclusion": "tecnica_deficiente"},

    # --- REGLAS DE NIVEL 2 (Encadenadas) ---
    {"condiciones": ["falla_cobertura_gas", "porosidad_visible"], "conclusion": "diagnostico: porosidad severa por perdida de gas."},
    {"condiciones": ["parametro_excedido", "avance_rapido"], "conclusion": "diagnostico: socavado severo por alta corriente."},
    {"condiciones": ["contaminacion_base", "falla_cobertura_gas"], "conclusion": "diagnostico: rechazo de pieza por contaminacion."},
    {"condiciones": ["tecnica_deficiente", "salpicadura_excesiva"], "conclusion": "diagnostico: inconsistencia operativa y tecnica pobre."}
]


# MOTOR DE INFERENCIA (Con Encadenamiento)


def encadenamiento_hacia_adelante(hechos_iniciales):
    hechos_conocidos = set(hechos_iniciales)
    conclusiones_finales = []
    cambio = True

    while cambio:
        cambio = False
        for regla in REGLAS:
            if all(cond in hechos_conocidos for cond in regla["condiciones"]):
                if regla["conclusion"] not in hechos_conocidos:
                    hechos_conocidos.add(regla["conclusion"])
                    cambio = True 
                    if regla["conclusion"].startswith("diagnostico:"):
                        conclusiones_finales.append(regla["conclusion"].replace("diagnostico: ", ""))

    return conclusiones_finales


# EXTRACCIÓN DE HECHOS


def extraer_hechos(texto):
    texto = texto.lower()
    hechos = []

    if "inestable" in texto or "parpadea" in texto: hechos.append("arco_inestable")
    if "poco gas" in texto or "sin gas" in texto: hechos.append("gas_insuficiente")
    if "salpicadura" in texto or "chispas" in texto: hechos.append("salpicadura_excesiva")
    if "corriente alta" in texto or "mucho amperaje" in texto: hechos.append("corriente_alta")
    if "poros" in texto or "burbujas" in texto: hechos.append("porosidad_visible")
    if "sucio" in texto or "oxido" in texto: hechos.append("material_sucio")
    if "irregular" in texto or "torcido" in texto: hechos.append("cordon_irregular")
    if "rapido" in texto or "rápido" in texto: hechos.append("avance_rapido")

    return hechos


# INTERFAZ DE ENTRADA (VOZ / TECLADO) Y SÍNTESIS


def hablar(texto):
    subprocess.run(["python", "fig/Notebook/tts.py", texto])

def reconocer_voz():
    r = sr.Recognizer()
    with sr.Microphone() as source:
        hablar("Describa los defectos de la soldadura.")
        print("\n[🎙️ Micrófono abierto] Hable ahora...")
        r.adjust_for_ambient_noise(source)
        audio = r.listen(source)

    try:
        texto = r.recognize_google(audio, language="es-ES")
        print("Texto reconocido:", texto)
        return texto
    except sr.UnknownValueError:
        print("No se pudo entender el audio.")
        return None
    except sr.RequestError:
        print("Error de conexión.")
        return None

def capturar_sintomas():
    print("SISTEMA EXPERTO: DIAGNÓSTICO DE SOLDADURA")
    opcion = input("¿Desea ingresar los síntomas usando el micrófono? (s/n): ").strip().lower()
    
    if opcion == 's':
        return reconocer_voz()
    else:
        hablar("Por favor, escriba los defectos detectados en la consola.")
        texto = input("\n[⌨️ Teclado] Escriba los síntomas detectados: ")
        return texto


# PROGRAMA PRINCIPAL
# Captura de síntomas (voz o teclado)
texto = capturar_sintomas()

if texto:
    hechos = extraer_hechos(texto)

    if not hechos:
        print("\nNo se detectaron sintomas clave.")
        hablar("No se detectaron sintomas clave.")
    else:
        print("\nHechos detectados internamente:", hechos)
        conclusiones = encadenamiento_hacia_adelante(hechos)

        if conclusiones:
            print("\n--- DIAGNÓSTICO FINAL ---")
            for c in conclusiones:
                print("->", c)
                hablar(c)
        else:
            print("\nDiagnóstico no concluyente.")
            hablar("Diagnóstico no concluyente.")

SISTEMA EXPERTO: DIAGNÓSTICO DE SOLDADURA

[🎙️ Micrófono abierto] Hable ahora...
Texto reconocido: material base muy sucio el arco parpadea bastante porque hay sin gas y la pieza quedó llena de

Hechos detectados internamente: ['arco_inestable', 'gas_insuficiente', 'material_sucio']

Diagnóstico no concluyente.
